In [ ]:
# Package
!pip install bertopic sentence-transformers kiwipiepy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

!find /content/drive/MyDrive -name "cnp_processed.csv" 2>/dev/null

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/cnp_processed.csv")
print(f" {len(df)}건")
print(df.columns.tolist())

In [ ]:
import ast
import warnings
warnings.filterwarnings("ignore")
from kiwipiepy import Kiwi
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

## Filtering step for CNP
CNP_KEYWORDS = {
    "차앤박", "CNP", "프로폴리스", "앰플", "PDRN",
    "안티포어", "트러블", "카밍", "클렌징", "세럼", "더마"
}
df_cnp = df[df["raw_text"].apply(
    lambda x: any(kw in str(x) for kw in CNP_KEYWORDS)
)].copy().reset_index(drop=True)
print(f" {len(df_cnp)}")

# Prepare for documents
docs = df_cnp["raw_text"].fillna("").tolist()
docs = [d for d in docs if len(d) >= 10]
print(f"{len(docs)}")

# Embedding model
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")



# Tokenizer for Korean
kiwi = Kiwi()

def korean_tokenizer(text):
    try:
        result = kiwi.analyze(text)
        tokens = [
            token.form for token in result[0][0]
            if token.tag in ("NNG", "NNP") and len(token.form) >= 2
        ]
        return tokens if tokens else text.split()
    except:
        return text.split()

vectorizer = CountVectorizer(
    tokenizer=korean_tokenizer,
    min_df=3,
    max_df=0.85,
    max_features=5000
)

## BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    representation_model=KeyBERTInspired(),
    nr_topics="auto",
    min_topic_size=15,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)
print(f"\n{len(set(topics)) - 1}")

In [ ]:
## Summary
topic_info = topic_model.get_topic_info()
print(topic_info[topic_info["Topic"] != -1][["Topic", "Count", "Name"]].to_string(index=False))

print("\n Keywords ")
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    keywords = topic_model.get_topic(topic_id)
    kw_str = ", ".join([kw for kw, _ in keywords[:8]])
    count = topics.count(topic_id)
    print(f"Topic {topic_id} ({count}): {kw_str}")

In [ ]:
import json

## LDA × BERTopic 교차검증
LDA_TOPICS = {
    "블랙헤드_안티포어": ["블랙", "헤드", "안티포어", "클리어", "키트"],
    "프로폴리스_앰플": ["프로폴리스", "앰플", "미스트", "에너지"],
    "트러블_이탈": ["트러블", "여드름", "자극", "카밍"],
    "더마코스메틱": ["더마", "피부과", "성분", "추출물"],
}

consensus = []
for lda_name, lda_kws in LDA_TOPICS.items():
    print(f"[ LDA: {lda_name} ]")
    found = False
    for topic_id in sorted(set(topics)):
        if topic_id == -1:
            continue
        bert_kws = [kw for kw, _ in topic_model.get_topic(topic_id)[:10]]
        overlap = set(lda_kws) & set(bert_kws)
        if overlap:
            count = topics.count(topic_id)
            print(f"  BERTopic Topic {topic_id} ({count}) | {overlap}")
            consensus.append({
                "lda_topic": lda_name,
                "bertopic_id": topic_id,
                "overlap_keywords": list(overlap),
                "bertopic_count": count,
                "confidence": "high"
            })
            found = True
    if not found:
        print(f"  No BERTopic (LDA signal)")
        consensus.append({
            "lda_topic": lda_name,
            "bertopic_id": None,
            "overlap_keywords": [],
            "bertopic_count": 0,
            "confidence": "low"
        })
    print()




topic_info_filtered = topic_info[topic_info["Topic"] != -1]
topic_info_filtered.to_csv("cnp_bertopic_results.csv",
                            index=False, encoding="utf-8-sig")

df_cnp_result = df_cnp.iloc[:len(topics)].copy()
df_cnp_result["bertopic_id"] = topics
df_cnp_result.to_csv("cnp_bertopic_documents.csv",
                      index=False, encoding="utf-8-sig")

topic_keywords = {}
for tid in sorted(set(topics)):
    if tid == -1:
        continue
    topic_keywords[str(tid)] = [kw for kw, _ in topic_model.get_topic(tid)[:10]]

with open("cnp_bertopic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(topic_keywords, f, ensure_ascii=False, indent=2)

consensus_df = pd.DataFrame(consensus)
consensus_df.to_csv("cnp_lda_bertopic_consensus.csv",
                     index=False, encoding="utf-8-sig")

print("  cnp_bertopic_results.csv")
print("  cnp_bertopic_documents.csv")
print("  cnp_bertopic_keywords.json")
print("  cnp_lda_bertopic_consensus.csv")

from google.colab import files
files.download("cnp_bertopic_results.csv")
files.download("cnp_bertopic_documents.csv")
files.download("cnp_bertopic_keywords.json")
files.download("cnp_lda_bertopic_consensus.csv")